<h1 dir="rtl" style="text-align: right;">
تحلیل اکتشافی داده های اضطراب اجتماعی
</h1>

<p dir="rtl" style="text-align: right;">
<strong>فاز دوم: <span dir="ltr">EDA</span></strong>
</p>

<p dir="rtl" style="text-align: right;">
اعضای تیم: علی خوش اخلاق، محمدحسین میرمعصومی، آرمین نورمحمدی، علی کریمی، محسن منصف
</p>

<h2 dir="rtl" style="text-align: right;">
1. کتابخانه ها و تنظیمات
</h2>

<p dir="rtl" style="text-align: right;">
از پایتون 3.13 استفاده کنید و ورژن های زیر
</p>

In [ ]:
# pandas==3.0.5 numpy==2.2.6 plotly==7.1.0 scipy==1.16.3 statsmodels==0.15.0 matplotlib==3.11.2

In [ ]:
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import matplotlib.pyplot as plt
import scipy
import statsmodels
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.stats.proportion import proportion_confint

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pio.renderers.default = "notebook"
px.defaults.template = "plotly_dark"
px.defaults.height = 450
BLUE = "#4C78A8"
RED = "#E45756"
TARGET = "Anxiety Level (1-10)"
LABEL = "Target"

<h2 dir="rtl" style="text-align: right;">
2. داده های اولیه
</h2>

In [ ]:
df = pd.read_csv("social_anxiety_dataset.csv")
print(df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe(include="all").T

In [ ]:
pd.DataFrame({
    'Missing': df.isnull().sum(),
    'Missing_%': (df.isna().mean() * 100).round(1),
    'unique': df.nunique(),
})

In [ ]:
print(df.duplicated().sum())

<h2 dir="rtl" style="text-align: right;">
مشاهدات اولیه
</h2>

<p dir="rtl" style="text-align: right;">
• سطر هامون 2030 تاس و ستون هامون 22 تاس
</p>

<p dir="rtl" style="text-align: right;">
• 9 تا ستون با داده گمشده داریم
</p>

<p dir="rtl" style="text-align: right;">
&nbsp;&nbsp;&nbsp;&nbsp;◦ که <span dir="ltr">Therapy History</span> با 90% داده گمشده بدترینه
</p>

<p dir="rtl" style="text-align: right;">
• <span dir="ltr">Sleep Hours</span>، <span dir="ltr">Physical Activity</span> و <span dir="ltr">Alcohol</span> مقدار منفی دارند
</p>

<p dir="rtl" style="text-align: right;">
• <span dir="ltr">Stress Level</span> مقدار 15 دارد که منطقی نیست
</p>

<p dir="rtl" style="text-align: right;">
• دوتا ستون <span dir="ltr">Target</span> و <span dir="ltr">is_Anxious</span> تغریبا یکسانند
</p>

<p dir="rtl" style="text-align: right;">
• ستون <span dir="ltr">Heart Rate</span> و <span dir="ltr">Caffeine</span> مقدار خارج از بازه دارند
</p>

<h2 dir="rtl" style="text-align: right;">
3. تحلیل و پاک سازی داده
</h2>

<h3 dir="rtl" style="text-align: right;">
3.1 جمعیت شناختی:
<span dir="ltr">Age, Gender, Occupation</span>
</h3>

<h3 dir="rtl" style="text-align: right;">
3.2 سبک زندگی:
<span dir="ltr">Sleep Hours, Physical Activity, Caffeine Intake, Alcohol Consumption, Smoking, Diet Quality</span>
</h3>

<h3 dir="rtl" style="text-align: right;">
3.3 فیزیولوژیک:
<span dir="ltr">Heart Rate, Breathing Rate, Sweating Level, Dizziness</span>
</h3>

<h4 dir="rtl" style="text-align: right;">
روش بررسی متغیرهای فیزیولوژیک
</h4>

<p dir="rtl" style="text-align: right;">
در این بخش، نوع داده، مقادیر گمشده، دامنه مقادیر و داده‌های پرت چهار متغیر
<span dir="ltr">Heart Rate</span>،
<span dir="ltr">Breathing Rate</span>،
<span dir="ltr">Sweating Level</span>
و
<span dir="ltr">Dizziness</span>
بررسی می‌شوند. برای متغیرهای عددی از روش
<span dir="ltr">IQR</span>
و برای متغیرهای ترتیبی و دسته‌ای از بررسی مقادیر یکتا و فراوانی استفاده می‌شود.
</p>

In [ ]:
# Create a copy and validate physiological variables

physiological_df = df[
    [
        "Heart Rate (bpm)",
        "Breathing Rate (breaths/min)",
        "Sweating Level (1-5)",
        "Dizziness"
    ]
].copy()

description = physiological_df[
    [
        "Heart Rate (bpm)",
        "Breathing Rate (breaths/min)",
        "Sweating Level (1-5)"
    ]
].describe()

display(description.T)

# Checking if numerical columns are actually numeric and with no missing values
for col in physiological_df.columns[:3]:
    physiological_df[col] = pd.to_numeric(
        physiological_df[col],
        errors="coerce"
    )

    print(
        col,
        "=>",
        physiological_df[col].isna().sum(),
        "non-numeric or missing values"
    )

print()
print(physiological_df.dtypes)

In [ ]:
# Inspect upper values and detect heart rate outliers using IQR

heart_rate = physiological_df["Heart Rate (bpm)"]

print(
    heart_rate
    .value_counts()
    .sort_index()
    .tail(10)
)

q1 = heart_rate.quantile(0.25)
q3 = heart_rate.quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

heart_rate_outliers = physiological_df[
    (heart_rate < lower_bound) |
    (heart_rate > upper_bound)
]

print("\nQ1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower_bound)
print("Upper bound:", upper_bound)
print("Number of outliers:", len(heart_rate_outliers))

print(
    heart_rate_outliers["Heart Rate (bpm)"]
    .value_counts()
    .sort_index()
)

# Inspect related features for heart rate outliers

outlier_indices = heart_rate_outliers.index

outlier_context = df.loc[
    outlier_indices,
    [
        "Age",
        "Heart Rate (bpm)",
        "Breathing Rate (breaths/min)",
        "Sweating Level (1-5)",
        "Dizziness",
        "Stress Level (1-10)",
        "Anxiety Level (1-10)",
        "Medication"
    ]
]

display(outlier_context.head(10))

# Replace invalid heart rate values with the median

physiological_df.loc[
    physiological_df["Heart Rate (bpm)"] == 220,
    "Heart Rate (bpm)"
] = np.nan

heart_rate_median = physiological_df["Heart Rate (bpm)"].median()

physiological_df["Heart Rate (bpm)"] = (
    physiological_df["Heart Rate (bpm)"]
    .fillna(heart_rate_median)
)

print("Median used:", heart_rate_median)
print(
    "Remaining missing values:",
    physiological_df["Heart Rate (bpm)"].isna().sum()
)

In [ ]:
# Detect breathing rate outliers using IQR

breathing_rate = physiological_df["Breathing Rate (breaths/min)"]

q1_breathing = breathing_rate.quantile(0.25)
q3_breathing = breathing_rate.quantile(0.75)
iqr_breathing = q3_breathing - q1_breathing

lower_bound_breathing = q1_breathing - 1.5 * iqr_breathing
upper_bound_breathing = q3_breathing + 1.5 * iqr_breathing

breathing_outlier_mask = (
    (breathing_rate < lower_bound_breathing) |
    (breathing_rate > upper_bound_breathing)
)

breathing_rate_outliers = physiological_df[breathing_outlier_mask]

print("Q1:", q1_breathing)
print("Q3:", q3_breathing)
print("IQR:", iqr_breathing)
print("Lower bound:", lower_bound_breathing)
print("Upper bound:", upper_bound_breathing)
print("Number of outliers:", breathing_outlier_mask.sum())

print(
    breathing_rate_outliers["Breathing Rate (breaths/min)"]
    .value_counts()
    .sort_index()
)

In [ ]:
# Inspect sweating level values and frequencies

sweating_level = physiological_df["Sweating Level (1-5)"]

print(sweating_level.unique())

print(
    sweating_level
    .value_counts(dropna=False)
    .sort_index()
)

In [ ]:
# Inspect dizziness categories and frequencies

dizziness = physiological_df["Dizziness"]

print(dizziness.unique())
print(dizziness.value_counts(dropna=False))

<h4 dir="rtl" style="text-align: right;">
جمع‌بندی و مشاهدات متغیرهای فیزیولوژیک
</h4>

<p dir="rtl" style="text-align: right;">
• سه متغیر عددی با موفقیت به نوع عددی تبدیل شدند و هیچ مقدار غیرقابل‌تبدیل یا گمشده اولیه در آن‌ها مشاهده نشد.
</p>

<p dir="rtl" style="text-align: right;">
• در متغیر
<strong><span dir="ltr">Heart Rate</span></strong>
تعداد ۳۰ داده پرت شناسایی شد که همگی برابر با
<span dir="ltr">220 bpm</span>
بودند.
</p>

<p dir="rtl" style="text-align: right;">
• تکرار دقیق مقدار ۲۲۰، فاصله زیاد آن با سایر مقادیر و نبود الگوی مشخص در متغیرهای مرتبط، احتمال وجود خطای ثبت یا مقدار نامعتبر سیستماتیک را تقویت می‌کند.
</p>

<p dir="rtl" style="text-align: right;">
• در متغیر
<strong><span dir="ltr">Breathing Rate</span></strong>
هیچ داده پرتی با روش
<span dir="ltr">IQR</span>
شناسایی نشد.
</p>

<p dir="rtl" style="text-align: right;">
• متغیر
<strong><span dir="ltr">Sweating Level</span></strong>
فقط شامل مقادیر معتبر ۱ تا ۵ و متغیر
<strong><span dir="ltr">Dizziness</span></strong>
فقط شامل دسته‌های معتبر
<span dir="ltr">Yes</span>
و
<span dir="ltr">No</span>
بود.
</p>

<h4 dir="rtl" style="text-align: right;">
اقدامات انجام‌شده
</h4>

<p dir="rtl" style="text-align: right;">
• مقادیر نامعتبر ۲۲۰ در ستون
<span dir="ltr">Heart Rate</span>
ابتدا به
<span dir="ltr">NaN</span>
تبدیل و سپس با <strong>میانه مقادیر معتبر</strong> جایگزین شدند.
</p>

<p dir="rtl" style="text-align: right;">
• جایگزینی با میانه باعث حفظ تعداد ردیف‌ها و جلوگیری از ایجاد مقادیر مصنوعی خارج از توزیع اصلی شد؛ با این حال، ممکن است فراوانی داده‌ها را در اطراف میانه کمی افزایش دهد.
</p>

<p dir="rtl" style="text-align: right;">
• متغیرهای
<span dir="ltr">Breathing Rate</span>،
<span dir="ltr">Sweating Level</span>
و
<span dir="ltr">Dizziness</span>
به دلیل نداشتن مقدار نامعتبر، بدون تغییر باقی ماندند.
</p>

<h3 dir="rtl" style="text-align: right;">
3.4 درمان، سابقه و متغیر هدف:<br>

<span dir="ltr">Family History, Medication, Therapy Sessions,</span><br>

<span dir="ltr">Therapy History, Recent Major Life Event, Stress Level, Anxiety Level, Target</span>
</h3>

<h3 dir="rtl" style="text-align: right;">
3.5 ادغام پاک سازی ها و ساخت
<span dir="ltr">clean_df</span>
</h3>

<h2 dir="rtl" style="text-align: right;">
4. ویژوال تک متغیره
</h2>

<h3 dir="rtl" style="text-align: right;">
4.1 جمعیت شناختی
</h3>

<h3 dir="rtl" style="text-align: right;">
4.2 سبک زندگی
</h3>

<h3 dir="rtl" style="text-align: right;">
4.3 فیزیولوژیک
</h3>

<h3 dir="rtl" style="text-align: right;">
4.4 درمان، سابقه و متغیر هدف
</h3>

<h2 dir="rtl" style="text-align: right;">
5. ویژوال دومتغیره
</h2>

<h3 dir="rtl" style="text-align: right;">
5.1 ستون های دسته ای با اضطراب
</h3>

<h3 dir="rtl" style="text-align: right;">
5.2 ستون های عددی با اضطراب
</h3>

<h2 dir="rtl" style="text-align: right;">
6. آزمون های آماری
</h2>

<h3 dir="rtl" style="text-align: right;">
6.1 آزمون همبستگی: عددی با عددی
</h3>

<h3 dir="rtl" style="text-align: right;">
6.2 آزمون
<span dir="ltr">t-test</span> و
<span dir="ltr">ANOVA</span>:
دسته ای با عددی
</h3>

<h3 dir="rtl" style="text-align: right;">
6.3 آزمون
<span dir="ltr">chi-square</span>:
دسته ای با دسته ای
</h3>

<h3 dir="rtl" style="text-align: right;">
6.4 امتیازی: بازه های اطمینان
</h3>

<h2 dir="rtl" style="text-align: right;">
7.
<span dir="ltr">KPI</span>
و فیچرهای تعاملی
</h2>

<h3 dir="rtl" style="text-align: right;">
7.1 استخراج
<span dir="ltr">KPI</span>
و فیچرهای تعاملی از ستون ها
</h3>

<h3 dir="rtl" style="text-align: right;">
7.2 ویژوال
<span dir="ltr">KPI</span>
و فیچرهای جدید
</h3>

<h2 dir="rtl" style="text-align: right;">
8. ویژوال چندمتغیره: ترکیب ویژگی ها و گروه های پرخطر
</h2>

<h2 dir="rtl" style="text-align: right;">
9. امتیازی: ویژوال سه متغیره و بیشتر
</h2>